In [ ]:
!pip install pandas

In [ ]:
import pandas as pd 
df=pd.read_csv("all_kindle_review.csv")

In [ ]:
df.head()

In [ ]:
df=df[['reviewText','rating']]

In [ ]:
df

In [ ]:
df.shape

In [ ]:
## Missing Values
df.isnull().sum()

In [ ]:
df['rating'].unique()

In [ ]:
df['rating'].value_counts()

In [ ]:
## Preprocessing and cleaning
## Positive review is 1 and Negetive review is 0
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

In [ ]:
df['rating'].unique()

In [ ]:
df['rating'].value_counts()

In [ ]:
## Lower all the cases
df['reviewText'] = df['reviewText'].str.lower()

In [ ]:
df.head()

In [ ]:
!pip install nltk

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already available
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))  # Load English stopwords

def clean_text(text):
    # 1. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)  
    
    # 2. Remove HTML Tags
    text = re.sub(r'<.*?>', '', text)  
    
    # 3. Remove Special Characters (but keep spaces)
    text = re.sub(r'[^a-z A-Z 0-9\s]', '', text)  
    
    # 4. Remove Stopwords
    text = " ".join([word for word in text.split() if word.lower() not in stop_words])
    
    # 5. Remove Extra Spaces
    text = re.sub(r'\s+', ' ', text).strip()  
    
    return text

# Apply the function to the DataFrame
df['reviewText'] = df['reviewText'].apply(clean_text)


In [ ]:
df.head()

In [ ]:
import nltk
nltk.download('wordnet')

In [ ]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer=WordNetLemmatizer()

In [ ]:
def lemmatize_word(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [ ]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_word(x))

In [ ]:
!pip install scikit-learn

In [ ]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.2)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [ ]:
X_train_bow

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow,Y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,Y_train)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [ ]:
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [ ]:
print(accuracy_score(Y_test,y_pred_bow))

In [ ]:
print(accuracy_score(Y_test,y_pred_tfidf))

In [ ]:
print(confusion_matrix(Y_test,y_pred_bow))

In [ ]:
!pip install gensim

In [ ]:
from gensim.models import Word2Vec
import numpy as np

# Train Word2Vec model on X_train
word2vec_model = Word2Vec(sentences=X_train, vector_size=100, window=5, min_count=1, workers=4)

# Function to convert sentences to word embeddings (average of word vectors)
def sentence_to_vec(text, model):
    vectors = [model.wv[word] for word in text if word in model.wv]
    return sum(vectors) / len(vectors) if vectors else np.zeros(model.vector_size)

# Convert X_train and X_test into Word2Vec embeddings
X_train_w2v = np.array([sentence_to_vec(sentence, word2vec_model) for sentence in X_train])
X_test_w2v = np.array([sentence_to_vec(sentence, word2vec_model) for sentence in X_test])


In [ ]:
nb_model_w2v = GaussianNB().fit(X_train_w2v,Y_train)

In [ ]:
y_pred_w2v=nb_model_w2v.predict(X_test_w2v)

In [ ]:
print(accuracy_score(Y_test,y_pred_w2v))